## Extrair os dados

In [1]:
# carregar o arquivo escolas10KLocDif.sql
from pathlib import Path

caminho_sql = Path("escolas10KLocDif.sql")

texto = caminho_sql.read_text(encoding="utf-8")

print(type(texto))
print(len(texto))
print(texto[:500])

<class 'str'>
2097798
DROP TABLE IF EXISTS telefone;
DROP TABLE IF EXISTS escola;

CREATE TABLE escola (
  codigo INT PRIMARY KEY,
  nome 	 TEXT NOT NULL,
  cod_municipio	 INT NOT NULL,
  endereco TEXT NULL,
  compl_endereco TEXT NULL,
  bairro TEXT NULL,
  cod_loc_dif INT NULL,
  FOREIGN KEY (cod_municipio) REFERENCES municipio(codigo),
  FOREIGN KEY (cod_loc_dif) REFERENCES localizacao_dif(codigo)
);


INSERT INTO escola (codigo, nome, cod_municipio, endereco, compl_endereco, bairro, cod_loc_dif) VALUES (11022558, 


In [2]:
#transformar o texto em uma lista de linhas
linhas = texto.splitlines()
print(type(linhas))
print(len(linhas))
print(linhas[:5])

<class 'list'>
10001
['DROP TABLE IF EXISTS telefone;', 'DROP TABLE IF EXISTS escola;', '', 'CREATE TABLE escola (', '  codigo INT PRIMARY KEY,']


In [3]:
# Depois filtre apenas os INSERT da tabela escola
inserts = [linha for linha in linhas if linha.startswith("INSERT INTO escola")]
print(type(inserts))
print(len(inserts))
print(inserts[:5])

<class 'list'>
9985
["INSERT INTO escola (codigo, nome, cod_municipio, endereco, compl_endereco, bairro, cod_loc_dif) VALUES (11022558, 'EIEEF HAP BITT TUPARI', 1100015,  'TERRA INDIGENA RIO BRANCO', 'ALDEIA COLORADO', 'RURAL',2);", "INSERT INTO escola (codigo, nome, cod_municipio, endereco, compl_endereco, bairro, cod_loc_dif) VALUES (11024275, 'CEEJA LUIZ VAZ DE CAMOES', 1100015,  'AVENIDA RIO DE JANEIRO', 'ESCOLA', 'CIDADE ALTA',0);", "INSERT INTO escola (codigo, nome, cod_municipio, endereco, compl_endereco, bairro, cod_loc_dif) VALUES (11024291, 'EMMEF 7 DE SETEMBRO', 1100015,  'LINHA 60 COM A 140', NULL, NULL ,0);", "INSERT INTO escola (codigo, nome, cod_municipio, endereco, compl_endereco, bairro, cod_loc_dif) VALUES (11024372, 'EMEIEF ANA NERY', 1100015,  'ROLIM DE MOURA DO GUAPORE', NULL, NULL ,0);", "INSERT INTO escola (codigo, nome, cod_municipio, endereco, compl_endereco, bairro, cod_loc_dif) VALUES (11024666, 'EMEIEF BOA ESPERANCA', 1100015,  'LINHA P 50 KM 22', 'ZONA RURA

In [4]:
# Extrair os valores dos inserts usando expressões regulares
import re   
valores = []
for insert in inserts:
    match = re.search(r"VALUES\s*\((.*)\);", insert)
    if match:
        valores.append(match.group(1)) 
print(type(valores))
print(len(valores))
print(valores[:5])

<class 'list'>
9985
["11022558, 'EIEEF HAP BITT TUPARI', 1100015,  'TERRA INDIGENA RIO BRANCO', 'ALDEIA COLORADO', 'RURAL',2", "11024275, 'CEEJA LUIZ VAZ DE CAMOES', 1100015,  'AVENIDA RIO DE JANEIRO', 'ESCOLA', 'CIDADE ALTA',0", "11024291, 'EMMEF 7 DE SETEMBRO', 1100015,  'LINHA 60 COM A 140', NULL, NULL ,0", "11024372, 'EMEIEF ANA NERY', 1100015,  'ROLIM DE MOURA DO GUAPORE', NULL, NULL ,0", "11024666, 'EMEIEF BOA ESPERANCA', 1100015,  'LINHA P 50 KM 22', 'ZONA RURAL', 'LINHA P.50',0"]


# 

In [5]:
# ler valores com modulo csv
import csv

linhas_campos = list(
    csv.reader(
        valores,
        delimiter=",",
        quotechar="'",
        skipinitialspace=True
    )
)
campos = linhas_campos[0]

print(campos)
print(len(campos))
print(len(linhas_campos))

['11022558', 'EIEEF HAP BITT TUPARI', '1100015', 'TERRA INDIGENA RIO BRANCO', 'ALDEIA COLORADO', 'RURAL', '2']
7
9985


In [6]:
# Criar uma função de limpeza de campo retirar os espaços em branco e NULLs
def limpar_campo(campo):
    campo = campo.strip()  # Remove espaços em branco
    if campo.upper() == "NULL":
        return None  # Converte 'NULL' para None
    return campo
# Aplicar a função de limpeza em cada campo
linhas_limpa = []
for linha in linhas_campos:
    linha_limpa = [limpar_campo(campo) for campo in linha]
    linhas_limpa.append(linha_limpa)
print(linhas_limpa[:5])


[['11022558', 'EIEEF HAP BITT TUPARI', '1100015', 'TERRA INDIGENA RIO BRANCO', 'ALDEIA COLORADO', 'RURAL', '2'], ['11024275', 'CEEJA LUIZ VAZ DE CAMOES', '1100015', 'AVENIDA RIO DE JANEIRO', 'ESCOLA', 'CIDADE ALTA', '0'], ['11024291', 'EMMEF 7 DE SETEMBRO', '1100015', 'LINHA 60 COM A 140', None, None, '0'], ['11024372', 'EMEIEF ANA NERY', '1100015', 'ROLIM DE MOURA DO GUAPORE', None, None, '0'], ['11024666', 'EMEIEF BOA ESPERANCA', '1100015', 'LINHA P 50 KM 22', 'ZONA RURAL', 'LINHA P.50', '0']]


In [9]:
#Verificar se todas as linhas têm 7 campos
tamanhos = []

for linha in linhas_limpa:
    tamanhos.append(len(linha))

set(tamanhos)

{7}

In [10]:
# criar o DataFrame com pandas com as colunas   codigo, nome, cod_municipio,
#    endereco, compl_endereco,bairro, cod_loc_dif
import pandas as pd
colunas = [
    "codigo",
    "nome",
    "cod_municipio",
    "endereco",
    "compl_endereco",
    "bairro",
    "cod_loc_dif"
]
df = pd.DataFrame(linhas_limpa, columns=colunas)
print(df.head())
print(df.info())

     codigo                      nome cod_municipio  \
0  11022558     EIEEF HAP BITT TUPARI       1100015   
1  11024275  CEEJA LUIZ VAZ DE CAMOES       1100015   
2  11024291       EMMEF 7 DE SETEMBRO       1100015   
3  11024372           EMEIEF ANA NERY       1100015   
4  11024666      EMEIEF BOA ESPERANCA       1100015   

                    endereco   compl_endereco       bairro cod_loc_dif  
0  TERRA INDIGENA RIO BRANCO  ALDEIA COLORADO        RURAL           2  
1     AVENIDA RIO DE JANEIRO           ESCOLA  CIDADE ALTA           0  
2         LINHA 60 COM A 140              NaN          NaN           0  
3  ROLIM DE MOURA DO GUAPORE              NaN          NaN           0  
4           LINHA P 50 KM 22       ZONA RURAL   LINHA P.50           0  
<class 'pandas.DataFrame'>
RangeIndex: 9985 entries, 0 to 9984
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   codigo          9985 non-null   str  
 1   no

In [12]:
# converer as colunas codigo, cod_municipio e cod_loc_dif para numero  
df["codigo"] = pd.to_numeric(df["codigo"], errors="coerce").astype("Int64")
df["cod_municipio"] = pd.to_numeric(df["cod_municipio"], errors="coerce").astype("Int64")
df["cod_loc_dif"] = pd.to_numeric(df["cod_loc_dif"], errors="coerce").astype("Int64")
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 9985 entries, 0 to 9984
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   codigo          9985 non-null   Int64
 1   nome            9985 non-null   str  
 2   cod_municipio   9985 non-null   Int64
 3   endereco        9982 non-null   str  
 4   compl_endereco  5485 non-null   str  
 5   bairro          6578 non-null   str  
 6   cod_loc_dif     8715 non-null   Int64
dtypes: Int64(3), str(4)
memory usage: 575.4 KB
None
